# Soccer player and ball tracking

### Set path directory

In [1]:
from pathlib import Path
local_dir = Path.cwd() / "data"
local_dir.mkdir(parents=True, exist_ok=True)

### Import SoccerNet

In [2]:
import SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader

mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=str(local_dir))
mySoccerNetDownloader.password = "s0cc3rn3t"

### Download data

Uncomment to download data

In [3]:
mySoccerNetDownloader.downloadDataTask(task="tracking-2023", split=["train", "test", "challenge"])
mySoccerNetDownloader.downloadDataTask(task="jersey-2023", split=["train","test","challenge"])

### Manually unZIP files

## Visualise football clips

### tracking-2023

In [13]:
import os, cv2, time
from pathlib import Path
from collections import defaultdict

folder_name = "SNMOT-060"

def find_folder_downwards(target_folder: str, start_dir: Path | None = None) -> Path | None:
    start_dir = start_dir or local_dir
    for path in start_dir.rglob(target_folder):
        if path.is_dir():
            return path
    return None

# --- Locate sequence ---
file_dir = find_folder_downwards(folder_name)
if file_dir is None:
    raise FileNotFoundError(f"Folder '{folder_name}' not found from {local_dir}")

seqdir = Path(file_dir)
imgdir = seqdir / "img1"
gt_dir = seqdir / "gt"
gt_txt = gt_dir / "gt.txt"

# --- Build per-frame GT (if available) ---
per_frame = defaultdict(list)
if gt_txt.is_file():
    with open(gt_txt) as f:
        for line in f:
            if not line.strip():
                continue
            fr, tid, x, y, w, h, conf, *rest = line.strip().split(",")
            cls_ = int(float(rest[0])) if rest else 1
            per_frame[int(float(fr))].append(
                (int(float(tid)), cls_,
                 int(float(x)), int(float(y)), int(float(w)), int(float(h)))
            )

# --- Playback settings ---
win = "SNMOT preview"
target_fps = 25
frame_period = 1.0 / target_fps

# OpenCV image-sequence reader; expects 000001.jpg, 000002.jpg, ...
cap = cv2.VideoCapture(str(imgdir / "%06d.jpg"))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open image sequence at {imgdir}")

window_created = False
i = 1

try:
    while True:
        t0 = time.perf_counter()

        ok, img = cap.read()
        if not ok or img is None:
            break

        # Draw boxes if GT exists
        if per_frame:
            for tid, cls_, x, y, w, h in per_frame.get(i, []):
                color = (225, 225, 0)
                cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv2.putText(img, f"{tid}", (x, max(0, y - 4)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)

        # Create window when first frame is ready
        if not window_created:
            cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)
            window_created = True

        cv2.imshow(win, img)

        # Adaptive delay to hit target FPS
        elapsed = time.perf_counter() - t0
        remaining = frame_period - elapsed
        delay_ms = 1 if remaining <= 0 else int(remaining * 1000)

        key = cv2.waitKey(delay_ms) & 0xFF
        if key in (27, ord('q')):  # ESC or q
            break

        i += 1
finally:
    cap.release()
    cv2.destroyWindow(win)
    cv2.waitKey(1)  # let macOS process the close

[ WARN:0@399.848] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 62316.471838 ms


KeyboardInterrupt: 

### jersey-2023

In [10]:
import tempfile
import re
import numpy as np

data_split = "train"    # "train", "test", "challenge"
folder_number = "97"   # 0 - 1425 (challenge), 1210 (test), train (1426)

seqdir = Path(local_dir)
imgdir = seqdir / "jersey-2023" / data_split / "images" / folder_number


# Grab all matching images and sort them by frame number
images = sorted(imgdir.glob(f"{folder_number}_*.jpg"),
                key=lambda p: int(p.stem.split("_")[1]))


# --- Playback settings ---
win = "Clip preview"
target_fps = 25
frame_period = 1.0 / target_fps

cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)

try:
    for i, img_path in enumerate(images, start=1):
        t0 = time.perf_counter()
        img = cv2.imread(str(img_path))
        if img is None:
            continue


        cv2.imshow(win, img)

        elapsed = time.perf_counter() - t0
        remaining = frame_period - elapsed
        delay_ms = 1 if remaining <= 0 else int(remaining * 1000)

        key = cv2.waitKey(delay_ms) & 0xFF
        if key in (27, ord('q')):
            break
finally:
    cv2.destroyWindow(win)
    cv2.waitKey(1)

## Focus on tracking

### Ball identification example

In [12]:
## Ball pseudo-labeling & visualization

### Heuristic ball identification

"""
We don't have explicit ball labels in tracking-2023 `gt.txt`. Strategy per sequence:
1. Load all track boxes for that sequence.
2. Compute box area and aspect ratio per track per frame.
3. Aggregate statistics: median area, area IQR, mean speed (pixel/frame) for each track.
4. Ball candidates (can be multiple per sequence, max 3):
   - Have small median area (below 30th percentile AND absolute max 2000 px²).
   - Aspect ratio closer to square (roundness > 0.7).
   - Reasonable presence (not just 1-2 frames).
We return top 3 tracks by weighted score (small + round + present), subject to strict filters.
This handles: in-play ball, reserve balls from sideline, goal kicks, but excludes players.
"""

from pathlib import Path

import numpy as np, json, cv2, statistics

from collections import defaultdict

def find_folder_downwards(target_folder: str, start_dir: Path | None = None) -> Path | None:
    start_dir = start_dir or local_dir
    for path in start_dir.rglob(target_folder):
        if path.is_dir():
            return path
    return None


def load_gt(gt_path: Path):

    tracks = defaultdict(list)  # tid -> list of (frame, x,y,w,h)

    with open(gt_path) as f:

        for line in f:

            if not line.strip():

                continue

            fr, tid, x, y, w, h, conf, *_ = line.strip().split(',')

            fr_i = int(float(fr)); tid_i = int(float(tid))

            box = (float(x), float(y), float(w), float(h))

            tracks[tid_i].append((fr_i, *box))

    return tracks



def track_stats(track_boxes):

    # track_boxes: list of (frame, x,y,w,h)

    areas = []

    centers = []

    frames = []

    for fr,x,y,w,h in track_boxes:

        areas.append(w*h)

        centers.append((x + w/2.0, y + h/2.0))

        frames.append(fr)

    # speed: mean center displacement between consecutive frames present

    centers_sorted = [c for _,c in sorted(zip(frames, centers))]

    dists = []

    for (x1,y1),(x2,y2) in zip(centers_sorted, centers_sorted[1:]):

        dists.append(np.hypot(x2-x1, y2-y1))

    mean_speed = float(np.mean(dists)) if dists else 0.0

    med_area = statistics.median(areas) if areas else 0.0

    aspect_ratios = [ (w/h) if h>0 else 0 for _,_,_,w,h in track_boxes ]

    roundness = 1.0 - min(1.0, abs( statistics.median(aspect_ratios) - 1.0 ))  # closer to 1 => more round

    presence_ratio = len(track_boxes) / ( (max(frames)-min(frames)+1) if frames else 1 )

    return {

        'median_area': med_area,

        'mean_speed': mean_speed,

        'roundness': roundness,

        'presence_ratio': presence_ratio,

        'n_obs': len(track_boxes)

    }



def choose_ball_tracks(tracks, min_obs=3, max_balls=3, area_pct_thresh=30, abs_area_max=2000, min_roundness=0.7):

    """Return list of up to max_balls track IDs that look like balls."""

    stats = {tid: track_stats(boxes) for tid, boxes in tracks.items()}

    if not stats:

        return [], {}

    all_med_areas = np.array([s['median_area'] for s in stats.values()])

    all_speeds = np.array([s['mean_speed'] for s in stats.values()])

    all_presence = np.array([s['presence_ratio'] for s in stats.values()])

    # Normalize (avoid divide-by-zero)

    def norm(v):

        v = np.array(v, dtype=float)

        rng = (v.max() - v.min()) or 1.0

        return (v - v.min())/rng  # 0 = min, 1 = max

    # Inverse area score: smaller area => higher score

    area_norm = norm(all_med_areas)

    inv_area_score = 1.0 - area_norm  # larger original => smaller score

    # Presence score: higher presence => higher score

    presence_score = norm(all_presence)

    # Speed score: normalized (but low weight)

    speed_score = norm(all_speeds)

    # Roundness score

    roundness_scores = np.array([s['roundness'] for s in stats.values()])

    round_score = norm(roundness_scores)

    # Weighted combination: prioritize small area + high presence + roundness

    combo = 0.50*inv_area_score + 0.30*presence_score + 0.15*round_score + 0.05*speed_score

    # Strict filters

    area_thresh = np.percentile(all_med_areas, area_pct_thresh)

    ordered_tids = list(stats.keys())

    candidates = []

    for idx, tid in enumerate(ordered_tids):

        s = stats[tid]

        # Hard filters

        if s['median_area'] > min(area_thresh, abs_area_max):

            continue  # too large (either relative or absolute)

        if s['n_obs'] < min_obs:

            continue  # too few observations

        if s['roundness'] < min_roundness:

            continue  # not square enough

        candidates.append((tid, combo[idx]))

    # Sort by score descending, take top max_balls

    candidates.sort(key=lambda x: x[1], reverse=True)

    ball_tids = [tid for tid, _ in candidates[:max_balls]]

    return ball_tids, stats



# Example usage on one sequence (adjust folder_name)

folder_name = "SNMOT-121"  # change as needed

seq_path = find_folder_downwards(folder_name, local_dir)

gt_path = seq_path / 'gt' / 'gt.txt'

tracks = load_gt(gt_path)

ball_tids, stats = choose_ball_tracks(tracks)

print(f"Heuristic ball track ids: {ball_tids}")

for tid, s in list(stats.items())[:5]:

    print(tid, s)



### Visualize candidate ball tracks (all)

if ball_tids:

    imgdir = seq_path / 'img1'

    # Build per-frame boxes for all candidates

    ball_boxes = defaultdict(list)

    for ball_tid in ball_tids:

        for fr,x,y,w,h in tracks[ball_tid]:

            ball_boxes[fr].append((x,y,w,h,ball_tid))

    cap = cv2.VideoCapture(str(imgdir / "%06d.jpg"))

    if cap.isOpened():

        i=1

        win = "Ball heuristic preview"

        cv2.namedWindow(win, cv2.WINDOW_AUTOSIZE)

        try:

            while True:

                ok,img = cap.read()

                if not ok or img is None: break

                for x,y,w,h,tid in ball_boxes.get(i, []):

                    cv2.rectangle(img,(int(x),int(y)),(int(x+w),int(y+h)),(0,0,255),2)

                    cv2.putText(img, f"ball? {tid}", (int(x), max(0,int(y-5))), cv2.FONT_HERSHEY_SIMPLEX, 0.4,(0,0,255),1, cv2.LINE_AA)

                cv2.imshow(win,img)

                if cv2.waitKey(1) & 0xFF in (27, ord('q')): break

                i+=1

        finally:

            cap.release(); cv2.destroyWindow(win); cv2.waitKey(1)

else:

    print("No ball candidates identified; refine heuristics or label manually.")


Heuristic ball track ids: [16]
1 {'median_area': 18447.0, 'mean_speed': 5.608334081213885, 'roundness': 0.4299065420560748, 'presence_ratio': 1.0, 'n_obs': 144}
2 {'median_area': 11412.0, 'mean_speed': 6.323941516785463, 'roundness': 0.4398795180722892, 'presence_ratio': 0.76, 'n_obs': 570}
3 {'median_area': 11154.0, 'mean_speed': 7.206906777997328, 'roundness': 0.48, 'presence_ratio': 0.44533333333333336, 'n_obs': 334}
4 {'median_area': 9714.5, 'mean_speed': 6.032395379697196, 'roundness': 0.4285714285714286, 'presence_ratio': 0.9093333333333333, 'n_obs': 682}
5 {'median_area': 11319.0, 'mean_speed': 6.349169596078516, 'roundness': 0.546448087431694, 'presence_ratio': 0.9106666666666666, 'n_obs': 683}
